# E-Commerce Lakehouse: Bronze Order Items

## Load the source order items

Read the original Olist order items CSV from the source Volume.
Preserve source values as strings before applying Silver transformations.

In [ ]:
raw_order_items_df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/ecommerce/raw_files/olist_order_items_dataset.csv")
)

display(raw_order_items_df.limit(5))

## Inspect the source structure and row count

Count the order item records and inspect their data types.
Establish a baseline before saving the Bronze table.

In [ ]:
print("Source order item rows:", raw_order_items_df.count())
raw_order_items_df.printSchema()

## Add ingestion metadata

Add ingestion_timestamp to record when the order item data is
processed. Preserve all original source columns and values.

In [ ]:
from pyspark.sql.functions import current_timestamp

bronze_order_items_df = raw_order_items_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)

display(bronze_order_items_df.limit(5))

## Refresh and validate Bronze order items

Rebuild the Bronze table from the complete source dataset.
Check for 112,650 source rows before replacing existing contents,
then verify the saved row count.

In [ ]:
table_name = "workspace.ecommerce.bronze_order_items"

source_count = raw_order_items_df.count()

if source_count != 112650:
    raise ValueError(
        f"Expected 112,650 source items; found {source_count}. "
        "Stopped before replacing the table."
    )

(
    bronze_order_items_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_bronze_order_items_df = spark.table(table_name)
saved_count = saved_bronze_order_items_df.count()

if saved_count != source_count:
    raise ValueError("Saved Bronze item count does not match the source.")

print("Bronze order items refreshed:", saved_count)